# Workbook for analyzing combinations of witnesses in Caleb Mutch's apparatus criticus.

I'm interested in finding out whether the family sigla are being used to the utmost advantage. My hunch is that there are some opportunities to make better use of the family sigla, but I should test that hunch. That's the purpose of this notebook.

In the cells below, I read in a version of 

In [1]:
from lxml import etree
from pathlib import Path

In [2]:
NS = {"tei": "http://www.tei-c.org/ns/1.0"}

tree = etree.parse(Path("cutedition.xml"))

interesting = {
    "text",
    "front",
    "body",
    "back",
    "div",
    "listWit",
    "listBibl",
}

for el in tree.xpath("//tei:text//*", namespaces=NS):
    name = etree.QName(el).localname

    if name not in interesting:
        continue

    depth = sum(etree.QName(a).localname in interesting for a in el.iterancestors())

    attrs = []

    if el.get("type"):
        attrs.append(f'type="{el.get("type")}"')

    xml_id = el.get("{http://www.w3.org/XML/1998/namespace}id")
    if xml_id:
        attrs.append(f'xml:id="{xml_id}"')

    details = " " + " ".join(attrs) if attrs else ""

    print(f"{'  ' * depth}{name}{details}")

  front xml:id="front"
    div type="section" xml:id="preface"
      div type="section" xml:id="preface-section-1"
        div type="section" xml:id="preface-subsection-1.1"
        div type="section" xml:id="preface-subsection-1.2"
      div type="section" xml:id="preface-section-2"
        div type="section" xml:id="preface-subsection-2.1"
        div type="section" xml:id="preface-subsection-2.2"
          div type="section" xml:id="bibliography-manuscripts"
            listWit xml:id="α"
              listWit
              listWit
            listWit xml:id="κ"
              listWit
            listWit
              listWit
              listWit
          div type="section" xml:id="bibliography-early-editions"
            listWit xml:id="early-editions"
        div type="section" xml:id="preface-subsection-2.3"
        div type="section" xml:id="preface-subsection-2.4"
        div type="section" xml:id="preface-subsection-2.5"
        div type="section" xml:id="preface-subsection-2

In [3]:
import pandas as pd
rows = []

for app in tree.xpath("//tei:app", namespaces=NS):
    app_id = app.get("{http://www.w3.org/XML/1998/namespace}id")

    for reading in app.xpath("./tei:lem | ./tei:rdg", namespaces=NS):
        rows.append(
            {
                "app_id": app_id,
                "type": etree.QName(reading).localname,
                "reading": " ".join(reading.itertext()).strip(),
                "wit": reading.get("wit"),
                "source": reading.get("source"),
            }
        )

df = pd.DataFrame(rows)

In [4]:
df

,app_id,type,reading,wit,source
0,NaN,lem,PROPTER QUID DOLENTES ET DELECTANTES FISTULANT...,NaN,None
1,NaN,rdg,PROPTER QUID DOLENTES,#β #Ea #Vm #Vo #Vz,None
2,NaN,rdg,PROPTER QUID DOLENTES ET GAUDENTES,#Vb #Vo,None
3,NaN,lem,mathematicam,NaN,None
4,NaN,rdg,mathematica,#Ma #Nu #Vm,None
...,...,...,...,...,...
5969,NaN,rdg,procurari,#γ,None
5970,NaN,lem,Nunc,NaN,None
5971,NaN,rdg,Non,#Ea #Px #Vm #Vz,None
5972,NaN,lem,quam,NaN,None


In [5]:
df["wit"].value_counts()

wit
#π             749
#Vo            274
#γ             177
#π #Ea         122
#λ             118
              ... 
#π #κ            1
#Ea #Py #Pz      1
#Py #Va #Vz      1
#Ea #Ma #Vm      1
#Py #Vm #Vo      1
Name: count, Length: 436, dtype: int64

In [6]:
df["witnesses"] = df["wit"].fillna("").str.split()

In [7]:
df["n_witnesses"] = df["witnesses"].str.len()

In [8]:
witness_report = df.sort_values("n_witnesses", ascending=False)[
    ["app_id", "type", "reading", "witnesses", "n_witnesses"]
]

In [9]:
witness_report["witness_set"] = witness_report["witnesses"].apply(frozenset)

In [10]:
witness_report["witness_set"].value_counts().head(50)

witness_set
frozenset({})                      2765
frozenset({#π})                     749
frozenset({#Vo})                    274
frozenset({#γ})                     177
frozenset({#π, #Ea})                124
frozenset({#λ})                     118
frozenset({#m, #Va, #v})            113
frozenset({#κ})                      64
frozenset({#β})                      63
frozenset({#α, #π})                  54
frozenset({#Vo, #Ea})                52
frozenset({#Gu, #Pp})                46
frozenset({#Px, #Vm})                33
frozenset({#Vm, #Vo})                32
frozenset({#Ma, #Vm})                30
frozenset({#Va, #Py})                29
frozenset({#Vb, #Vz, #β, #Ea})       28
frozenset({#π, #Py})                 24
frozenset({#Vz, #Ma})                24
frozenset({#Ea, #γ})                 23
frozenset({#α})                      23
frozenset({#Py, #Ea})                22
frozenset({#Vb, #m, #v})             20
frozenset({#Px, #Vz})                20
frozenset({#Px, #Ma})       

In [11]:
witness_report["type"].value_counts()

type
rdg    3208
lem    2766
Name: count, dtype: int64

In [12]:
groups = {}

for group in tree.xpath("//tei:listWit[@xml:id]", namespaces=NS):
    group_id = group.get("{http://www.w3.org/XML/1998/namespace}id")

    witnesses = group.xpath(".//tei:witness/@xml:id", namespaces=NS)

    groups[group_id] = frozenset(witnesses)

In [13]:
groups

{'α': frozenset({'Ea', 'Nu', 'Py', 'Va', 'Vb', 'β', 'γ'}),
 'κ': frozenset({'Ma', 'Mb', 'Px', 'Vm', 'Vz', 'λ'}),
 'early-editions': frozenset({'m', 'v'})}

In [14]:
mss_groups = {
    "α": frozenset({"Ea", "Nu", "Py", "Va", "Vb"}),
    "β": frozenset({"Nu", "Py"}),
    "γ": frozenset({"Va", "Vb"}),
    "κ": frozenset({"Ma", "Mb", "Px", "Vm", "Vz"}),
    "λ": frozenset({"Ma", "Vm", "Vz", "Mb"}),
    "j": frozenset({"Gu", "Pp", "Pu", "Pz", "Vo", "Ca", "Pa", "Pq", "Sc", "Vc"}),
    "π": frozenset({"Gu", "Pp", "Pu", "Pz", "Vo"})
}

In [15]:
def match_group(witness_set):
    # Try to match the witness set to one of the known groups in mss_groups
    for group_name, group_witnesses in mss_groups.items():
        if witness_set == group_witnesses:
            return group_name
        else:
            # Check if the witness set is a subset of the group witnesses
            if witness_set.issubset(group_witnesses):
                return f"{group_name} (subset)"

In [16]:
witness_report.columns

Index(['app_id', 'type', 'reading', 'witnesses', 'n_witnesses', 'witness_set'], dtype='str')

In [17]:
witness_report["witness_groups_match"] = witness_report["witness_set"].apply(match_group)

In [18]:
witness_report["witness_groups_match"].value_counts()

witness_groups_match
α (subset)    2765
Name: count, dtype: int64

In [19]:
def match_group(witness_set):
    # Look for exact matches first
    for group_name, group_witnesses in mss_groups.items():
        if witness_set == group_witnesses:
            return group_name

In [20]:
witness_report["witness_groups_match"] = witness_report["witness_set"].apply(
    match_group
)

In [21]:
witness_report["witness_groups_match"].value_counts()

Series([], Name: count, dtype: int64)

In [22]:
def expand_groups(witness_set):
    expanded = set()

    for witness in witness_set:
        witness = witness.lstrip("#")
        if witness in mss_groups:
            expanded.update(mss_groups[witness])
        else:
            expanded.add(witness)

    return frozenset(expanded)

In [23]:
witness_report["expanded_witness_set"] = witness_report["witness_set"].apply(
    expand_groups
)

In [24]:
witness_report["expanded_witness_groups_match"] = witness_report["expanded_witness_set"].apply(
    match_group
)

In [25]:
witness_report["expanded_witness_groups_match"].value_counts()

expanded_witness_groups_match
π    750
γ    177
λ    118
κ     64
β     63
α     23
Name: count, dtype: int64

In [26]:
def group_already_used(row):
    if row["expanded_witness_groups_match"] is None:
        return False
    else:
        group = "#" + str(row["expanded_witness_groups_match"])
    return group in row["witness_set"]

witness_report["group_already_used"] = witness_report.apply(group_already_used, axis=1)

In [27]:
witness_report["group_already_used"].value_counts()

group_already_used
False    4780
True     1194
Name: count, dtype: int64

In [28]:
def compare_to_groups(witness_set):
    results = {}

    for group_name, group_witnesses in mss_groups.items():
        missing = group_witnesses - witness_set
        extra = witness_set - group_witnesses

        results[group_name] = {
            "missing": missing,
            "extra": extra,
        }

    return results

In [29]:
witness_report["compare_to_groups"] = witness_report["expanded_witness_set"].apply(
    compare_to_groups
)

In [30]:
witness_report["compare_to_groups"].value_counts()

compare_to_groups
{'α': {'missing': frozenset({'Va', 'Ea', 'Vb', 'Py', 'Nu'}), 'extra': frozenset({})}, 'β': {'missing': frozenset({'Py', 'Nu'}), 'extra': frozenset({})}, 'γ': {'missing': frozenset({'Va', 'Vb'}), 'extra': frozenset({})}, 'κ': {'missing': frozenset({'Vz', 'Px', 'Vm', 'Mb', 'Ma'}), 'extra': frozenset({})}, 'λ': {'missing': frozenset({'Vz', 'Vm', 'Mb', 'Ma'}), 'extra': frozenset({})}, 'j': {'missing': frozenset({'Gu', 'Vo', 'Vc', 'Pq', 'Pa', 'Pz', 'Ca', 'Pp', 'Sc', 'Pu'}), 'extra': frozenset({})}, 'π': {'missing': frozenset({'Gu', 'Vo', 'Pz', 'Pp', 'Pu'}), 'extra': frozenset({})}}                                                                                                                2765
{'α': {'missing': frozenset({'Ea', 'Nu', 'Va', 'Vb', 'Py'}), 'extra': frozenset({'Gu', 'Vo', 'Pz', 'Pp', 'Pu'})}, 'β': {'missing': frozenset({'Py', 'Nu'}), 'extra': frozenset({'Gu', 'Vo', 'Pz', 'Pp', 'Pu'})}, 'γ': {'missing': frozenset({'Va', 'Vb'}), 'extra': frozenset({'Gu', 'Vo',

In [31]:
def group_distance(witness_set, group_witnesses):
    missing = group_witnesses - witness_set
    extra = witness_set - group_witnesses

    return len(missing) + len(extra)

In [32]:
witness_report.columns

Index(['app_id', 'type', 'reading', 'witnesses', 'n_witnesses', 'witness_set',
       'witness_groups_match', 'expanded_witness_set',
       'expanded_witness_groups_match', 'group_already_used',
       'compare_to_groups'],
      dtype='str')

In [33]:
unique_expanded_witness_sets = witness_report["expanded_witness_set"].unique()

In [34]:
len(unique_expanded_witness_sets)

426

In [35]:
for witness_set in unique_expanded_witness_sets:
    print(f"Witness set: {witness_set}")
    for group_name, group_witnesses in mss_groups.items():
        distance = group_distance(witness_set, group_witnesses)
        print(f"  Distance to group {group_name}: {distance}")

Witness set: frozenset({'Gu', 'Vo', 'Pz', 'Vb', 'Ea', 'Pu', 'Nu'})
  Distance to group α: 6
  Distance to group β: 7
  Distance to group γ: 7
  Distance to group κ: 12
  Distance to group λ: 11
  Distance to group j: 9
  Distance to group π: 4
Witness set: frozenset({'Pu', 'Gu', 'Vo', 'Ea', 'Ma', 'Va', 'Px', 'Pz', 'Pp', 'Vb', 'Vm'})
  Distance to group α: 10
  Distance to group β: 13
  Distance to group γ: 9
  Distance to group κ: 10
  Distance to group λ: 11
  Distance to group j: 11
  Distance to group π: 6
Witness set: frozenset({'Va', 'Vo', 'Pz', 'Ea', 'Pp', 'Pu'})
  Distance to group α: 7
  Distance to group β: 8
  Distance to group γ: 6
  Distance to group κ: 11
  Distance to group λ: 10
  Distance to group j: 8
  Distance to group π: 3
Witness set: frozenset({'Va', 'Px', 'Ea', 'Vm', 'Py', 'Vz'})
  Distance to group α: 5
  Distance to group β: 6
  Distance to group γ: 6
  Distance to group κ: 5
  Distance to group λ: 6
  Distance to group j: 16
  Distance to group π: 11
Witness s

In [36]:
witness_report["expanded_witness_set"].value_counts().head(50)

expanded_witness_set
frozenset({})                                          2765
frozenset({Gu, Vo, Pz, Pp, Pu})                         750
frozenset({Vo})                                         274
frozenset({Va, Vb})                                     177
frozenset({Gu, Vo, Pz, Pp, Ea, Pu})                     124
frozenset({Vz, Vm, Mb, Ma})                             118
frozenset({Va, v, m})                                   113
frozenset({Vz, Px, Vm, Mb, Ma})                          64
frozenset({Py, Nu})                                      63
frozenset({Pu, Gu, Vo, Ea, Nu, Va, Pz, Pp, Vb, Py})      54
frozenset({Vo, Ea})                                      52
frozenset({Gu, Pp})                                      46
frozenset({Vm, Px})                                      33
frozenset({Vm, Vo})                                      32
frozenset({Vm, Ma})                                      30
frozenset({Va, Py})                                      29
frozenset({Ea, Vb, 

In [37]:
witness_report.columns

Index(['app_id', 'type', 'reading', 'witnesses', 'n_witnesses', 'witness_set',
       'witness_groups_match', 'expanded_witness_set',
       'expanded_witness_groups_match', 'group_already_used',
       'compare_to_groups'],
      dtype='str')

In [38]:
exact_matches = witness_report[witness_report["witness_groups_match"].notna()].copy()

In [39]:
exact_matches[["witness_set", "expanded_witness_set", "witness_groups_match"]].head(25)

,witness_set,expanded_witness_set,witness_groups_match


In [40]:
def contained_groups(witness_set):
    matches = []

    for group_name, group_witnesses in mss_groups.items():
        if group_witnesses.issubset(witness_set):
            matches.append(group_name)

    return matches

In [41]:
witness_report["contained_groups"] = witness_report["expanded_witness_set"].apply(
    contained_groups
)

In [42]:
witness_report['contained_groups'].value_counts()

contained_groups
[]                    4060
[π]                   1025
[γ]                    287
[λ]                    154
[β]                    146
[κ, λ]                  76
[α, β, γ, π]            68
[α, β, γ]               45
[γ, π]                  23
[β, π]                  21
[β, γ]                  19
[λ, π]                  12
[γ, λ]                  11
[β, γ, π]                5
[κ, λ, π]                5
[α, β, γ, λ]             5
[γ, λ, π]                4
[β, λ]                   3
[α, β, γ, κ, λ, π]       2
[γ, κ, λ, π]             1
[β, λ, π]                1
[α, β, γ, κ, λ]          1
Name: count, dtype: int64

In [43]:
def subgroup_already_used(row):
    if row["contained_groups"] is None:
        return False
    else:
        return [("#" + str(group)) in row["witness_set"] for group in row["contained_groups"]]

In [44]:
witness_report['subgroup_already_used'] = witness_report.apply(subgroup_already_used, axis=1)

In [45]:
witness_report['subgroup_already_used'].value_counts()

subgroup_already_used
[]                                         4060
[True]                                     1611
[True, True]                                 89
[True, False]                                76
[True, False, False, True]                   73
[True, False, False]                         45
[True, True, True]                           10
[True, False, True]                           5
[True, False, False, True, False, True]       2
[False]                                       1
[True, True, False, True]                     1
[True, False, False, True, False]             1
Name: count, dtype: int64

In [46]:
def unused_contained_groups(row):
    unused = []

    for group in row["contained_groups"]:
        if f"#{group}" not in row["witness_set"]:
            unused.append(group)

    return unused

In [47]:
witness_report["unused_contained_groups"] = witness_report.apply(
    unused_contained_groups, axis=1
)

In [48]:
witness_report["unused_contained_groups"].value_counts()

unused_contained_groups
[]           5770
[β, γ]        118
[λ]            82
[β, γ, λ]       3
[π]             1
Name: count, dtype: int64

In [49]:
from itertools import combinations


def optimal_representation(witness_set):
    best_representation = set(witness_set)

    group_names = list(mss_groups)

    for n in range(1, len(group_names) + 1):
        for group_combo in combinations(group_names, n):

            covered = set()

            for group in group_combo:
                covered.update(mss_groups[group])

            # We can't use a group if it introduces witnesses
            # that aren't actually in the reading.
            if not covered.issubset(witness_set):
                continue

            # Anything not covered by groups must be represented
            # by its individual siglum.
            remaining = set(witness_set) - covered

            representation = set(group_combo) | remaining

            if len(representation) < len(best_representation):
                best_representation = representation

    return frozenset(best_representation)

In [50]:
witness_report["optimal_representation"] = witness_report["expanded_witness_set"].apply(
    optimal_representation
)

In [51]:
witness_report["actual_n_sigla"] = witness_report["witness_set"].apply(len)

In [52]:
witness_report["optimal_n_sigla"] = witness_report["optimal_representation"].apply(len)

In [53]:
witness_report["sigla_saved"] = (
    witness_report["actual_n_sigla"] - witness_report["optimal_n_sigla"]
)

In [54]:
witness_report.loc[
    witness_report["sigla_saved"] > 0,
    [
        "witness_set",
        "expanded_witness_set",
        "optimal_representation",
        "actual_n_sigla",
        "optimal_n_sigla",
        "sigla_saved",
    ],
].head(30)

,witness_set,expanded_witness_set,optimal_representation,actual_n_sigla,optimal_n_sigla,sigla_saved
937,"frozenset({#β, #λ, #Ea, #Vb, #Vz})","frozenset({Ea, Mb, Ma, Nu, Vb, Vm, Py, Vz})","frozenset({Ea, Vb, β, λ})",5,4,1
3307,"frozenset({#Gu, #Vo, #Pu, #Pp, #Pz})","frozenset({Gu, Vo, Pz, Pp, Pu})",frozenset({π}),5,1,4


In [55]:
witness_report["sigla_saved"].min()

np.int64(0)

In [56]:
witness_report["sigla_saved"].value_counts()

sigla_saved
0    5972
1       1
4       1
Name: count, dtype: int64

In [57]:
witness_report.columns

Index(['app_id', 'type', 'reading', 'witnesses', 'n_witnesses', 'witness_set',
       'witness_groups_match', 'expanded_witness_set',
       'expanded_witness_groups_match', 'group_already_used',
       'compare_to_groups', 'contained_groups', 'subgroup_already_used',
       'unused_contained_groups', 'optimal_representation', 'actual_n_sigla',
       'optimal_n_sigla', 'sigla_saved'],
      dtype='str')

In [58]:
witness_report.groupby("type")["sigla_saved"].sum()

type
lem    0
rdg    5
Name: sigla_saved, dtype: int64

In [59]:
candidates = witness_report[witness_report["sigla_saved"] > 0].copy()

In [60]:
candidates.columns

Index(['app_id', 'type', 'reading', 'witnesses', 'n_witnesses', 'witness_set',
       'witness_groups_match', 'expanded_witness_set',
       'expanded_witness_groups_match', 'group_already_used',
       'compare_to_groups', 'contained_groups', 'subgroup_already_used',
       'unused_contained_groups', 'optimal_representation', 'actual_n_sigla',
       'optimal_n_sigla', 'sigla_saved'],
      dtype='str')

In [61]:
candidates_reduced = candidates[
    [
        "app_id",
        "type",
        "reading",
        "witnesses",
        "optimal_representation",
        "actual_n_sigla",
        "optimal_n_sigla",
        "sigla_saved"
    ]
]

In [62]:
candidates_reduced

,app_id,type,reading,witnesses,optimal_representation,actual_n_sigla,optimal_n_sigla,sigla_saved
937,NaN,rdg,,"[#β, #λ, #Ea, #Vb, #Vz]","frozenset({Ea, Vb, β, λ})",5,4,1
3307,NaN,rdg,,"[#Gu, #Pp, #Pu, #Pz, #Vo]",frozenset({π}),5,1,4


In [63]:
def unpack_optimal_representation(optimal_representation):
    unpacked = [ms for ms in optimal_representation]
    return unpacked

In [64]:
candidates_reduced["proposed"] = candidates_reduced["optimal_representation"].apply(unpack_optimal_representation)

In [65]:
candidates_reduced

,app_id,type,reading,witnesses,optimal_representation,actual_n_sigla,optimal_n_sigla,sigla_saved,proposed
937,NaN,rdg,,"[#β, #λ, #Ea, #Vb, #Vz]","frozenset({Ea, Vb, β, λ})",5,4,1,"[Ea, Vb, β, λ]"
3307,NaN,rdg,,"[#Gu, #Pp, #Pu, #Pz, #Vo]",frozenset({π}),5,1,4,[π]


In [66]:
candidates_reduced["witness_no_hash"] = candidates_reduced["witnesses"].apply(lambda x: [w.lstrip("#") for w in x])

In [67]:
candidates_reduced

,app_id,type,reading,witnesses,optimal_representation,actual_n_sigla,optimal_n_sigla,sigla_saved,proposed,witness_no_hash
937,NaN,rdg,,"[#β, #λ, #Ea, #Vb, #Vz]","frozenset({Ea, Vb, β, λ})",5,4,1,"[Ea, Vb, β, λ]","[β, λ, Ea, Vb, Vz]"
3307,NaN,rdg,,"[#Gu, #Pp, #Pu, #Pz, #Vo]",frozenset({π}),5,1,4,[π],"[Gu, Pp, Pu, Pz, Vo]"


In [68]:
candidates_reduced["proposed_sorted"] = candidates_reduced["proposed"].apply(lambda x: sorted(x))

In [69]:
candidates_reduced

,app_id,type,reading,witnesses,optimal_representation,actual_n_sigla,optimal_n_sigla,sigla_saved,proposed,witness_no_hash,proposed_sorted
937,NaN,rdg,,"[#β, #λ, #Ea, #Vb, #Vz]","frozenset({Ea, Vb, β, λ})",5,4,1,"[Ea, Vb, β, λ]","[β, λ, Ea, Vb, Vz]","[Ea, Vb, β, λ]"
3307,NaN,rdg,,"[#Gu, #Pp, #Pu, #Pz, #Vo]",frozenset({π}),5,1,4,[π],"[Gu, Pp, Pu, Pz, Vo]",[π]


In [70]:
candidates_reduced["witness_no_hash_sorted"] = candidates_reduced["witness_no_hash"].apply(lambda x: sorted(x))

In [71]:
candidates_reduced

,app_id,type,reading,witnesses,optimal_representation,actual_n_sigla,optimal_n_sigla,sigla_saved,proposed,witness_no_hash,proposed_sorted,witness_no_hash_sorted
937,NaN,rdg,,"[#β, #λ, #Ea, #Vb, #Vz]","frozenset({Ea, Vb, β, λ})",5,4,1,"[Ea, Vb, β, λ]","[β, λ, Ea, Vb, Vz]","[Ea, Vb, β, λ]","[Ea, Vb, Vz, β, λ]"
3307,NaN,rdg,,"[#Gu, #Pp, #Pu, #Pz, #Vo]",frozenset({π}),5,1,4,[π],"[Gu, Pp, Pu, Pz, Vo]",[π],"[Gu, Pp, Pu, Pz, Vo]"


In [72]:
candidates_reduced.columns

Index(['app_id', 'type', 'reading', 'witnesses', 'optimal_representation',
       'actual_n_sigla', 'optimal_n_sigla', 'sigla_saved', 'proposed',
       'witness_no_hash', 'proposed_sorted', 'witness_no_hash_sorted'],
      dtype='str')

In [73]:
final_candidates = candidates_reduced[
    [
        "type",
        "reading",
        "witnesses",
        "witness_no_hash_sorted",
        "proposed_sorted",
        "actual_n_sigla",
        "optimal_n_sigla",
        "sigla_saved"
    ]
]

In [74]:
final_candidates = final_candidates.rename(columns={
    "witnesses": "original_witnesses",
    "witness_no_hash_sorted": "witnesses",
    "proposed_sorted": "proposed",
    "actual_n_sigla": "actual_sigla",
    "optimal_n_sigla": "optimal_sigla",
    "sigla_saved": "difference"
})

In [75]:
import csv
final_candidates.sort_values("difference", ascending=False, inplace=True)
final_candidates.to_csv("final_candidates.csv", index=False, quoting=csv.QUOTE_ALL)

In [102]:
final_candidates.groupby("type")["difference"].sum()

type
lem    82
rdg     7
Name: difference, dtype: int64

In [98]:
final_candidates.info()

<class 'pandas.DataFrame'>
Index: 52 entries, 126 to 2755
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   type           52 non-null     str   
 1   reading        52 non-null     str   
 2   witnesses      52 non-null     object
 3   witnesses      52 non-null     object
 4   proposed       52 non-null     object
 5   actual_sigla   52 non-null     int64 
 6   optimal_sigla  52 non-null     int64 
 7   difference     52 non-null     int64 
dtypes: int64(3), object(3), str(2)
memory usage: 3.7+ KB
